In [0]:
catalog = "workspace"
schema = "lakeflow_examples"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.lakeflow_examples;

### Generate employee_cdf data

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

employees_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("birth_date", DateType(), True),
    StructField("operation", StringType(), True),      # ["UPDATE", "INSERT", "DELETE", "TRUNCATE"]
    StructField("sequence_num", IntegerType(), True)
])

data = [
    (101, "Alice", "Smith", 28, date(1998, 5, 12), "INSERT", 1),
    (102, "Bob", "Jones", 35, date(1991, 9, 23), "INSERT", 1),
    (101, "Alice", "Smith", 29, date(1998, 5, 12), "UPDATE", 2),
    (103, "Charlie", "Brown", 42, date(1984, 11, 2), "INSERT", 1),
    (102, "Bob", "Jones", 35, date(1991, 9, 23), "DELETE", 2)
]

employees_cdf = spark.createDataFrame(data, schema=employees_schema)

employees_cdf.write\
    .format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{schema}.employees_cdf")

# display(employees_cdf)